# Exact permutation tests for benchmark contamination: paper experiments

**Kaggle settings:** GPU **T4** (one is enough), **Internet on**. Do **not** pip-install anything.

Run **one stage per session**. Set `STAGE` in the next cell, then *Save Version → Save & Run All (commit)*.
When it finishes, download `<stage>_results.zip` from the version's Output tab and send it to me.

| Stage | What it does | Estimated time |
|---|---|---|
| `audit1` | 6 public models (incl. Pythia controls) × 5 public benchmarks, published option orders | 3–5 h |
| `audit2` | 5 larger public models (1–2.8B) × 5 benchmarks | 5–8 h |
| `power_mmlu` | 3 models fine-tuned on 300 shuffled MMLU items; 6 exposure settings; 300 clean items | 3–5 h |
| `power_arc` | same on ARC-Challenge | 2–4 h |

Each unit of work is saved as soon as it finishes. If a session stops, run the same stage again; finished units are skipped.

In [ ]:
STAGE = "audit1"     # audit1 | audit2 | power_mmlu | power_arc
import os, shutil
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
OUT_DIR = f"{BASE}/{STAGE}_results"; os.makedirs(OUT_DIR, exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile core.py
"""Exact permutation tests for multiple-choice benchmark contamination (paper version).

For an item with n options and published order identity, every permutation p (p[j] = published index shown at slot j)
is scored by the total log-likelihood of the formatted item including the answer letter.
Three exact sub-tests (published order uniform on S_n, model independent of it):
  * order  : rank of the published order among all permutations (or identity + M random ones)      K = n! or M+1
  * anch   : rank among permutations that keep the correct answer in its published slot            K = (n-1)! or M+1
  * slot   : rank of the published answer slot among the n slots, each slot scored by the logsumexp
             over all permutations placing the correct answer there (exact enumeration only)         K = n
Under the null, slot and anch ranks are independent, so their Fisher combination is also exact.
"""
import itertools, math, random, time
import numpy as np
import torch
from scipy import stats, special

LET = "ABCDEFGHIJKLMNOP"


def fmt(q, opts, ans_pos):
    s = f"Question: {q.strip()}\n" + "".join(f"{LET[i]}. {str(o).strip()}\n" for i, o in enumerate(opts))
    return s + "Answer:", f" {LET[ans_pos]}"


def perm_plan(n, ans, rng, M=None):
    """Permutations to score and index lists for the three tests. Identity is index 0."""
    ident = tuple(range(n))
    if M is None:  # exact enumeration
        perms = [ident] + [p for p in itertools.permutations(range(n)) if p != ident]
        anch = [i for i, p in enumerate(perms) if p[ans] == ans]
        slots = [[i for i, p in enumerate(perms) if p.index(ans) == s] for s in range(n)]
        return perms, list(range(len(perms))), anch, slots
    perms = [ident] + [tuple(int(x) for x in rng.permutation(n)) for _ in range(M)]
    others = [i for i in range(n) if i != ans]
    anch = [0]
    for _ in range(M):
        o = list(rng.permutation(others)); p = [ans if j == ans else int(o.pop()) for j in range(n)]
        perms.append(tuple(p)); anch.append(len(perms) - 1)
    return perms, list(range(M + 1)), anch, None


def item_texts(item, perms):
    q, opts, a = item["q"], item["opts"], item["ans"]
    out = []
    for p in perms:
        pr, an = fmt(q, [opts[i] for i in p], p.index(a))
        out.append(pr + an)
    return out


@torch.no_grad()
def seq_logprob(model, tok, texts, dev, token_budget=2048, maxlen=1024):
    """Total log-probability of each text; dynamic batches of about token_budget tokens, sorted by length."""
    lens = [len(t) for t in texts]
    order = np.argsort(lens)[::-1]          # longest first: the first batch shows any memory problem at once
    res = np.zeros(len(texts)); i = 0
    while i < len(order):
        est = lens[order[i]] // 3 + 8       # conservative token estimate (about 3 characters per token)
        idx = order[i:i + max(1, token_budget // est)]
        enc = tok([texts[k] for k in idx], return_tensors="pt", padding=True, truncation=True, max_length=maxlen).to(dev)
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1]
        tgt = enc["input_ids"][:, 1:]
        lse = torch.logsumexp(logits.float(), -1)
        tl = logits.gather(-1, tgt[..., None])[..., 0].float()
        res[idx] = ((tl - lse) * enc["attention_mask"][:, 1:].float()).sum(1).cpu().numpy()
        del logits, lse, tl
        i += len(idx)
    return res


@torch.no_grad()
def mink_pp(model, tok, texts, dev, k=0.2, bs=2, maxlen=1024):
    """Min-K%++ (Zhang et al. 2024): mean of the lowest k fraction of standardized token log-probs. Higher = more member-like."""
    out = np.zeros(len(texts))
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=maxlen).to(dev)
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
        lsm = torch.log_softmax(logits, -1); del logits
        p = lsm.exp()
        mu = (p * lsm).sum(-1); sig = ((p * lsm * lsm).sum(-1) - mu ** 2).clamp_min(1e-8).sqrt(); del p
        tl = lsm.gather(-1, enc["input_ids"][:, 1:, None])[..., 0]; del lsm
        z = ((tl - mu) / sig).cpu().numpy(); m = enc["attention_mask"][:, 1:].cpu().numpy().astype(bool)
        for b in range(z.shape[0]):
            v = np.sort(z[b][m[b]]); n = max(1, int(len(v) * k)); out[i + b] = v[:n].mean()
    return out


def rank_u(scores_pub, scores_other, rng):
    """Randomized rank of the published score among {published} ∪ others. Returns (u in (0,1), integer rank, K)."""
    gt = (scores_other > scores_pub[:, None] + 1e-9).sum(1)
    tie = (np.abs(scores_other - scores_pub[:, None]) <= 1e-9).sum(1)
    r = gt + np.floor(rng.random(len(gt)) * (tie + 1)).astype(int) + 1
    K = scores_other.shape[1] + 1
    return (r - rng.random(len(gt))) / K, r, K


def score_bench(model, tok, items, dev, rng, M=None, token_budget=2048, log=print):
    """Score all permutations; return per-item dicts of u and r for order / anch / slot tests plus raw scores."""
    plans, texts, spans = [], [], []
    for it in items:
        perms, full, anch, slots = perm_plan(len(it["opts"]), it["ans"], rng, M)
        plans.append((perms, full, anch))
        spans.append((len(texts), len(texts) + len(perms)))
        texts += item_texts(it, perms)
    t = time.time()
    lp = seq_logprob(model, tok, texts, dev, token_budget)
    log(f"    {len(texts)} sequences in {time.time()-t:.0f}s")
    out = {"scores": [], "full_idx": [], "anch_idx": [], "M": M}
    for (perms, full, anch), (a, b) in zip(plans, spans):
        out["scores"].append(lp[a:b]); out["full_idx"].append(full); out["anch_idx"].append(anch)
    return out


def tests_from_scores(S, items, M, rng):
    """Compute u for order, anch, slot, and Fisher(slot, anch) from saved scores."""
    res = {k: [] for k in ["order", "anch", "slot", "slot_anch"]}
    Ks = {}
    order_pub, order_oth, anch_pub, anch_oth, slot_pub, slot_oth = [], [], [], [], [], []
    for s, it, fi, ai in zip(S["scores"], items, S["full_idx"], S["anch_idx"]):
        order_pub.append(s[0]); order_oth.append(s[[i for i in fi if i != 0]])
        anch_pub.append(s[0]); anch_oth.append(s[[i for i in ai if i != 0]])
        if M is None:
            n, a = len(it["opts"]), it["ans"]
            perms = [tuple(range(n))] + [p for p in itertools.permutations(range(n)) if p != tuple(range(n))]
            agg = np.array([special.logsumexp(s[[i for i, p in enumerate(perms) if p.index(a) == sl]]) for sl in range(n)])
            slot_pub.append(agg[a]); slot_oth.append(np.delete(agg, a))
    for name, pub, oth in [("order", order_pub, order_oth), ("anch", anch_pub, anch_oth), ("slot", slot_pub, slot_oth)]:
        if not pub:
            continue
        u, r, K = rank_u(np.array(pub), np.array(oth), rng)
        res[name] = {"u": u, "r": r, "K": K}
    if M is None:
        f = -2 * (np.log(res["slot"]["u"]) + np.log(res["anch"]["u"]))
        res["slot_anch"] = {"u": stats.chi2.sf(f, 4), "r": None, "K": None}  # per-item exact p-value, uniform under H0
    else:
        res.pop("slot"); res.pop("slot_anch")
    return res


# ---------------------------------------------------------------- set-level exact p-values
def p_meanrank_exact(r, K):
    N = len(r); conv = np.array([1.0]); pmf = np.ones(K) / K
    for _ in range(N):
        conv = np.convolve(conv, pmf)
    return float(conv[: int(r.sum()) - N + 1].sum())


def p_fisher(u):
    u = np.clip(u, 1e-300, 1)
    return float(stats.chi2.sf(-2 * np.log(u).sum(), 2 * len(u)))


def p_top1(r, K):
    return float(stats.binom.sf(np.sum(r == 1) - 1, len(r), 1.0 / K))


def set_tests(t):
    """Exact set-level p-values for one benchmark (all items) and each sub-test."""
    out = {}
    for name, d in t.items():
        if not d:
            continue
        u = d["u"]
        row = {"n": int(len(u)), "mean_u": float(np.mean(u)), "p_fisher": p_fisher(u),
               "p_ks_low": float(stats.kstest(u, "uniform", alternative="greater").pvalue)}
        if d.get("r") is not None:
            row.update({"K": int(d["K"]), "top1_rate": float(np.mean(d["r"] == 1)), "p_top1": p_top1(d["r"], d["K"]),
                        "p_meanrank": float(stats.norm.cdf((d["r"].mean() - (d["K"] + 1) / 2) / math.sqrt((d["K"] ** 2 - 1) / 12 / len(u))))})
        out[name] = row
    return out


def power_split(u_c, u_0, Ns=(25, 50, 100, 200), B=2000, seed=0, alpha=0.05, mk_c=None, mk_0=None, r_c=None, r_0=None, K=None):
    """Power on contaminated items, size on clean items, from disjoint resampling (with replacement) of each pool.
    Min-K%++ baseline uses the clean pool as its reference (an advantage the exact tests do not need)."""
    rng = np.random.default_rng(seed); out = {}
    for N in Ns:
        row = {"contaminated": {}, "clean": {}}
        for lab, uu, rr in [("contaminated", u_c, r_c), ("clean", u_0, r_0)]:
            ix = rng.integers(0, len(uu), (B, N))
            row[lab]["fisher"] = float(np.mean([p_fisher(uu[i]) < alpha for i in ix]))
            if rr is not None:
                row[lab]["top1"] = float(np.mean([p_top1(rr[i], K) < alpha for i in ix]))
        if mk_c is not None:
            ref = mk_0[rng.integers(0, len(mk_0), (B, N))].mean(1); thr = np.quantile(ref, 1 - alpha)
            row["contaminated"]["minkpp_ref"] = float(np.mean(mk_c[rng.integers(0, len(mk_c), (B, N))].mean(1) > thr))
        out[str(N)] = row
    return out


# ---------------------------------------------------------------- contamination simulation
def finetune(model, tok, items, dev, mode="full", epochs=1, lr=2e-5, bs=4, seed=0, log=print):
    """mode='full': loss on all tokens (pretraining-style). mode='answer': loss on the answer only (instruction-tuning style)."""
    torch.manual_seed(seed); rnd = random.Random(seed)
    model.train(); model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    data = [fmt(it["q"], it["opts"], it["ans"]) for it in items]
    for ep in range(epochs):
        rnd.shuffle(data)
        for i in range(0, len(data), bs):
            batch = data[i:i + bs]
            enc = tok([p + a for p, a in batch], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(dev)
            labels = enc["input_ids"].clone(); labels[enc["attention_mask"] == 0] = -100
            if mode == "answer":
                for b, (p, a) in enumerate(batch):
                    labels[b, :len(tok(p)["input_ids"])] = -100
            loss = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], labels=labels).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(set_to_none=True)
        log(f"    epoch {ep+1}: last loss {loss.item():.3f}")
    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    del opt
    return model


In [ ]:
%%writefile run.py
"""Staged experiments. Usage: python run.py <stage> <out_dir> [--fake]
Stages: audit1, audit2 (public models on public benchmarks, published orders)
        power_mmlu, power_arc (simulated contamination by fine-tuning; options re-shuffled by us)
Every unit of work is cached as one .npz in out_dir, so a stopped run resumes where it left off."""
import sys, os, json, time, gc, traceback
import numpy as np, torch
import core

STAGE, OUT = sys.argv[1], sys.argv[2]; FAKE = "--fake" in sys.argv
os.makedirs(OUT, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
log = lambda *a: print(*a, flush=True)

AUDIT_BENCH = {"mmlu": (500, None), "arc": (500, None), "obqa": (500, None), "mmlupro": (300, 29), "supergpqa": (300, 29)}
STAGES = {
    "audit1": {"models": ["EleutherAI/pythia-410m-deduped", "EleutherAI/pythia-1.4b-deduped", "HuggingFaceTB/SmolLM2-360M",
                          "Qwen/Qwen2.5-0.5B", "Qwen/Qwen3-0.6B-Base", "TinyLlama/TinyLlama_v1.1"]},
    "audit2": {"models": ["EleutherAI/pythia-2.8b-deduped", "HuggingFaceTB/SmolLM2-1.7B", "Qwen/Qwen2.5-1.5B",
                          "Qwen/Qwen3-1.7B-Base", "allenai/OLMo-2-0425-1B"]},
    "power_mmlu": {"bench": "mmlu", "models": ["HuggingFaceTB/SmolLM2-360M", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen3-0.6B-Base"]},
    "power_arc": {"bench": "arc", "models": ["HuggingFaceTB/SmolLM2-360M", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen3-0.6B-Base"]},
}
POWER_ITEMS = 600  # first half fine-tuned on (contaminated), second half held out (clean)
POWER_CONFIGS = [("base", 0), ("full", 1), ("full", 3), ("answer", 1), ("answer", 3), ("answer", 5)]
LR, BS_TRAIN = 2e-5, 4
if FAKE:
    AUDIT_BENCH = {"mmlu": (20, None), "mmlupro": (10, 5)}
    STAGES = {k: {**v, "models": ["fake"]} for k, v in STAGES.items()}
    POWER_ITEMS, POWER_CONFIGS, LR = 40, [("base", 0), ("full", 2), ("answer", 2)], 1e-3


# ---------------------------------------------------------------- data (published order kept unless shuffle=True)
def load_bench(name, n, seed=0, shuffle=False):
    if FAKE:
        from fake import fake_items
        return fake_items(n, 10 if name in ("mmlupro", "supergpqa") else 4, seed)
    from datasets import load_dataset
    rows = []
    if name == "mmlu":
        ds = load_dataset("cais/mmlu", "all", split="test")
        rows = [(r["question"], list(r["choices"]), int(r["answer"])) for r in ds]
    elif name in ("arc", "obqa"):
        ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test") if name == "arc" else load_dataset("allenai/openbookqa", "main", split="test")
        for r in ds:
            labs, txt = list(r["choices"]["label"]), list(r["choices"]["text"])
            if len(txt) == 4 and r["answerKey"] in labs:
                rows.append((r["question"] if name == "arc" else r["question_stem"], txt, labs.index(r["answerKey"])))
    elif name == "mmlupro":
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
        rows = [(r["question"], list(r["options"]), int(r["answer_index"])) for r in ds if len(r["options"]) == 10]
    elif name == "supergpqa":
        ds = load_dataset("m-a-p/SuperGPQA", split="train")
        for r in ds:
            opts = list(r["options"])
            if r.get("answer_letter") and len(opts) >= 4:
                a = "ABCDEFGHIJ".index(str(r["answer_letter"]).strip()[0])
            elif r.get("answer") in opts:
                a = opts.index(r["answer"])
            else:
                continue
            if len(opts) >= 4:
                rows.append((r["question"], opts, a))
    rng = np.random.default_rng(seed)
    pick = rng.permutation(len(rows))[:n]
    items = []
    for i in pick:
        q, o, a = rows[int(i)]
        if shuffle:
            p = rng.permutation(len(o)); o = [o[j] for j in p]; a = int(list(p).index(a))
        items.append({"q": q, "opts": o, "ans": a})
    return items


def load_model(name, dtype):
    if FAKE:
        from fake import fake_model
        return fake_model()
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(dev).eval()
    return m, tok


def budget(model):
    """Token budget per scoring batch so that fp32 log-sum-exp over the vocabulary stays near 1.2 GB."""
    V = model.get_output_embeddings().weight.shape[0]
    return int(max(256, min(8192, 3.0e8 / V)))


def save_scores(path, S, items, extra=None):
    np.savez(path, scores=np.array(S["scores"]), full_idx=np.array(S["full_idx"]), anch_idx=np.array(S["anch_idx"]),
             ans=np.array([it["ans"] for it in items]), nopt=np.array([len(it["opts"]) for it in items]),
             M=-1 if S["M"] is None else S["M"], **(extra or {}))


def score_safe(m, tok, items, M, seed=1):
    S = core.score_bench(m, tok, items, dev, np.random.default_rng(seed), M=M, token_budget=budget(m), log=log)
    if not np.isfinite(np.array(S["scores"])).all():
        log("    non-finite fp16 scores -> rescoring in fp32"); m = m.float()
        S = core.score_bench(m, tok, items, dev, np.random.default_rng(seed), M=M, token_budget=budget(m) // 2, log=log)
    return S, m


def free():
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()


# ---------------------------------------------------------------- audit stages
def run_audit(cfg):
    bench_items = {}
    for b, (n, M) in AUDIT_BENCH.items():
        try:
            bench_items[b] = load_bench(b, n); log(f"{b}: {len(bench_items[b])} items")
            json.dump(bench_items[b], open(f"{OUT}/items_{b}.json", "w"))
        except Exception as e:
            log(f"SKIPPING benchmark {b}: could not load ({e!r})")
    for name in cfg["models"]:
        short = name.split("/")[-1]
        todo = [b for b in bench_items if not os.path.exists(f"{OUT}/audit__{short}__{b}.npz")]
        if not todo:
            log("done:", short); continue
        try:
            m, tok = load_model(name, torch.float16 if dev == "cuda" else torch.float32)
        except Exception as e:
            log("FAILED to load", name, repr(e)); continue
        for b in todo:
            log(f"audit {short} / {b}")
            try:
                S, m = score_safe(m, tok, bench_items[b], AUDIT_BENCH[b][1])
                save_scores(f"{OUT}/audit__{short}__{b}.npz", S, bench_items[b])
            except Exception:
                log(traceback.format_exc())
            free()
        del m, tok; free()


# ---------------------------------------------------------------- power stages
def run_power(cfg):
    items = load_bench(cfg["bench"], POWER_ITEMS, seed=0, shuffle=True)
    half = len(items) // 2
    log(f"{cfg['bench']}: {len(items)} items ({half} contaminated, {len(items)-half} clean), options re-shuffled")
    for name in cfg["models"]:
        short = name.split("/")[-1]
        for mode, ep in POWER_CONFIGS:
            path = f"{OUT}/power__{short}__{cfg['bench']}__{mode}{ep}.npz"
            if os.path.exists(path):
                continue
            log(f"power {short} / {mode} x{ep}")
            try:
                if mode == "base":
                    m, tok = load_model(name, torch.float16 if dev == "cuda" else torch.float32)
                else:
                    m, tok = load_model(name, torch.float32)
                    t = time.time()
                    m = core.finetune(m, tok, items[:half], dev, mode=mode, epochs=ep, lr=LR, bs=BS_TRAIN, log=log)
                    log(f"    fine-tuned in {time.time()-t:.0f}s")
                    if dev == "cuda":
                        m = m.half()
                S, m = score_safe(m, tok, items, None)
                texts = ["".join(core.fmt(it["q"], it["opts"], it["ans"])) for it in items]
                mk = core.mink_pp(m, tok, texts, dev, bs=1 if not FAKE else 4)
                save_scores(path, S, items, {"mink": mk, "half": half})
            except Exception:
                log(traceback.format_exc())
            try:
                del m, tok
            except NameError:
                pass
            free()


# ---------------------------------------------------------------- quick summary of whatever is finished
def summarize():
    rng = np.random.default_rng(7); summ = {}
    for f in sorted(os.listdir(OUT)):
        if not f.endswith(".npz"):
            continue
        z = np.load(f"{OUT}/{f}", allow_pickle=True)
        items = [{"opts": list(range(int(n))), "ans": int(a)} for a, n in zip(z["ans"], z["nopt"])]
        M = None if int(z["M"]) < 0 else int(z["M"])
        S = {"scores": list(z["scores"]), "full_idx": list(z["full_idx"]), "anch_idx": list(z["anch_idx"])}
        t = core.tests_from_scores(S, items, M, rng)
        if f.startswith("audit"):
            summ[f[:-4]] = core.set_tests(t)
        else:
            h = int(z["half"]); row = {}
            for k, d in t.items():
                if not d:
                    continue
                rc = d["r"][:h] if d.get("r") is not None else None; r0 = d["r"][h:] if d.get("r") is not None else None
                row[k] = {"clean": core.set_tests({k: {**d, "u": d["u"][h:], "r": r0}})[k],
                          "contaminated": core.set_tests({k: {**d, "u": d["u"][:h], "r": rc}})[k],
                          "power": core.power_split(d["u"][:h], d["u"][h:], Ns=(25, 50, 100) if not FAKE else (10,), B=1000,
                                                    mk_c=z["mink"][:h], mk_0=z["mink"][h:], r_c=rc, r_0=r0, K=d.get("K"))}
            from sklearn.metrics import roc_auc_score
            y = np.r_[np.ones(h), np.zeros(len(z["mink"]) - h)]
            row["auroc"] = {"minkpp": float(roc_auc_score(y, z["mink"]))}
            for k in ["order", "slot", "anch"]:
                if t.get(k):
                    row["auroc"][k] = float(roc_auc_score(y, -t[k]["u"]))
            summ[f[:-4]] = row
    json.dump(summ, open(f"{OUT}/summary_{STAGE}.json", "w"), indent=1, default=float)
    for k, v in summ.items():
        if k.startswith("audit"):
            log(f"{k:55s} " + " | ".join(f"{t}: mean_u {d['mean_u']:.3f} p_fisher {d['p_fisher']:.2g}" for t, d in v.items()))
        else:
            log(f"{k:55s} AUROC {v['auroc']} | " + " | ".join(
                f"{t}: power@{list(v[t]['power'])[-1]} {v[t]['power'][list(v[t]['power'])[-1]]['contaminated']}" for t in ["order", "slot", "anch"] if t in v))


if __name__ == "__main__":
    t0 = time.time()
    (run_audit if STAGE.startswith("audit") else run_power)(STAGES[STAGE])
    summarize()
    log(f"stage {STAGE} finished in {(time.time()-t0)/60:.0f} min")


In [ ]:
!python run.py {STAGE} {OUT_DIR}

In [ ]:
shutil.make_archive(f"{BASE}/{STAGE}_results", "zip", OUT_DIR)
print("written:", f"{BASE}/{STAGE}_results.zip", round(os.path.getsize(f"{BASE}/{STAGE}_results.zip")/1e6, 2), "MB")